In [15]:
# Installation
# pip install yfinance pandas numpy ta streamlit
# pip install tabulate
# !pip install python-dotenv

In [16]:
#Read Config from env file
from dotenv import load_dotenv
import os

# Load the env file
load_dotenv("config.env")

# Access variables
NEWS_API_KEY = os.getenv("NEWS_API_KEY")
ALPHA_VANTAGE_KEY=os.getenv("ALPHA_VANTAGE_KEY")
FINNHUB_API_KEY=os.getenv("FINNHUB_API_KEY")

#Parameters

STOCKS = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS", "AXISBANK.NS", "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "BEL.NS", "BHARTIARTL.NS", "BPCL.NS", "BRITANNIA.NS", "CIPLA.NS", "COALINDIA.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS", "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS", "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "INDUSINDBK.NS", "INFY.NS", "ITC.NS", "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS", "M&M.NS", "MARUTI.NS", "NESTLEIND.NS", "NTPC.NS", "ONGC.NS", "POWERGRID.NS", "RELIANCE.NS", "SBILIFE.NS", "SBIN.NS", "SHRIRAMFIN.NS", "SUNPHARMA.NS", "TATACONSUM.NS", "TATAMOTORS.NS", "TATASTEEL.NS", "TCS.NS", "TECHM.NS", "TITAN.NS", "ULTRACEMCO.NS", "UPL.NS", "WIPRO.NS"
]

# STOCKS = ["BAJFINANCE.NS", "ADANIPORTS.NS", "JSWSTEEL.NS"]
INTERVAL = "5m"   # 1m, 5m, 15m
PERIOD = "5d"     # intraday history
OLLAMA_MODEL = "gemma3:4b"

CACHE_EXPIRY_MINUTES = 1500      # Refresh news every 15 minutes
MAX_ARTICLES         = 10      # Max articles to analyze per source
SENTIMENT_TRUNCATE   = 512     # FinBERT max token-safe char length

# Stock name aliases for better news matching (Nifty 50)
STOCK_ALIASES = {
    # Adani Group
    "ADANIENT":    ["Adani Enterprises", "Adani Enterprise", "Adani"],
    "ADANIPORTS":  ["Adani Ports", "Adani Port", "APSEZ"],
    
    # Financial Services
    "AXISBANK":    ["Axis Bank", "Axis"],
    "BAJAJ-AUTO":  ["Bajaj Auto", "Bajaj"],
    "BAJFINANCE":  ["Bajaj Finance", "Bajaj Finserv Finance"],
    "BAJAJFINSV":  ["Bajaj Finserv", "Bajaj Financial Services"],
    "HDFCBANK":    ["HDFC Bank", "HDFCBank", "HDFC"],
    "HDFCLIFE":    ["HDFC Life", "HDFC Life Insurance"],
    "ICICIBANK":   ["ICICI Bank", "ICICI"],
    "INDUSINDBK":  ["IndusInd Bank", "IndusInd"],
    "KOTAKBANK":   ["Kotak Mahindra Bank", "Kotak Bank", "Kotak"],
    "SBIN":        ["SBI", "State Bank of India", "State Bank"],
    "SBILIFE":     ["SBI Life", "SBI Life Insurance"],
    "SHRIRAMFIN":  ["Shriram Finance", "Shriram"],
    
    # IT Services
    "HCLTECH":     ["HCL Technologies", "HCL Tech", "HCL"],
    "INFY":        ["Infosys", "INFY"],
    "TCS":         ["TCS", "Tata Consultancy Services", "Tata Consultancy"],
    "TECHM":       ["Tech Mahindra", "Tech M"],
    "WIPRO":       ["Wipro", "Wipro Ltd"],
    
    # Pharmaceuticals
    "APOLLOHOSP":  ["Apollo Hospitals", "Apollo Hospital", "Apollo"],
    "CIPLA":       ["Cipla", "Cipla Ltd"],
    "DRREDDY":     ["Dr Reddy's", "Dr Reddy", "Dr. Reddy's Laboratories"],
    "SUNPHARMA":   ["Sun Pharma", "Sun Pharmaceutical", "Sun Pharmaceuticals"],
    
    # Consumer Goods
    "ASIANPAINT":  ["Asian Paints", "Asian Paint"],
    "BRITANNIA":   ["Britannia", "Britannia Industries"],
    "HINDUNILVR":  ["Hindustan Unilever", "HUL", "Unilever India"],
    "ITC":         ["ITC", "ITC Ltd"],
    "NESTLEIND":   ["Nestle India", "Nestle"],
    "TATACONSUM":  ["Tata Consumer", "Tata Consumer Products"],
    "TITAN":       ["Titan", "Titan Company"],
    
    # Automobiles
    "EICHERMOT":   ["Eicher Motors", "Eicher", "Royal Enfield"],
    "HEROMOTOCO":  ["Hero MotoCorp", "Hero Moto", "Hero"],
    "M&M":         ["Mahindra & Mahindra", "M&M", "Mahindra"],
    "MARUTI":      ["Maruti Suzuki", "Maruti", "Maruti Suzuki India"],
    "TATAMOTORS":  ["Tata Motors", "Tata Motor"],
    
    # Metals & Mining
    "COALINDIA":   ["Coal India", "Coal India Ltd"],
    "HINDALCO":    ["Hindalco", "Hindalco Industries"],
    "JSWSTEEL":    ["JSW Steel", "JSW"],
    "TATASTEEL":   ["Tata Steel", "Tata Steel Ltd"],
    
    # Energy & Oil/Gas
    "BPCL":        ["BPCL", "Bharat Petroleum", "Bharat Petroleum Corporation"],
    "NTPC":        ["NTPC", "NTPC Ltd"],
    "ONGC":        ["ONGC", "Oil and Natural Gas Corporation"],
    "POWERGRID":   ["Power Grid", "PowerGrid", "Power Grid Corporation"],
    "RELIANCE":    ["Reliance Industries", "Reliance", "RIL"],
    
    # Telecom
    "BHARTIARTL":  ["Bharti Airtel", "Airtel", "Bharti"],
    
    # Defense
    "BEL":         ["BEL", "Bharat Electronics", "Bharat Electronics Ltd"],
    
    # Conglomerate & Infrastructure
    "GRASIM":      ["Grasim Industries", "Grasim"],
    "LT":          ["Larsen & Toubro", "L&T", "Larsen and Toubro"],
    
    # Chemicals & Agriculture
    "UPL":         ["UPL", "UPL Ltd", "United Phosphorus"],
    
    # Cement
    "ULTRACEMCO":  ["UltraTech Cement", "UltraTech", "Ultratech"],
}


# Functions For Reading stock Data and All the Technical Analysis

In [17]:
# Intraday Stock Selection Pipeline (India)
# Rule-based model | Ready-to-run | Python 3
# Data Source: Yahoo Finance (for demo) -> Replace with Zerodha/Upstox API for real trading

# -------------------
# INSTALL DEPENDENCIES
# -------------------

import yfinance as yf
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator, ADXIndicator, MACD
from ta.volume import VolumeWeightedAveragePrice
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import AverageTrueRange, BollingerBands

############################
# Fetch Stock
############################
def fetch_data(symbol):
    df = yf.download(symbol, interval=INTERVAL, period=PERIOD, progress=False)
    # Fix for multi-index columns from yfinance
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[['Open','High','Low','Close','Volume']]
    df.dropna(inplace=True)
    return df

############################
# Compute Indicator
############################
def compute_indicators(df):
    close = df['Close'].astype(float)
    high = df['High'].astype(float)
    low = df['Low'].astype(float)
    volume = df['Volume'].astype(float)

    df['EMA9'] = EMAIndicator(close, window=9).ema_indicator()
    df['EMA21'] = EMAIndicator(close, window=21).ema_indicator()

    vwap = VolumeWeightedAveragePrice(high=high, low=low, close=close, volume=volume)
    df['VWAP'] = vwap.volume_weighted_average_price()

    df['RSI'] = RSIIndicator(close, window=14).rsi()
    
    # Stochastic Oscillator for additional momentum
    stoch = StochasticOscillator(high, low, close, window=14, smooth_window=3)
    df['Stoch_K'] = stoch.stoch()
    df['Stoch_D'] = stoch.stoch_signal()

    atr = AverageTrueRange(high, low, close, window=14)
    df['ATR'] = atr.average_true_range()

    # MACD for trend confirmation
    macd = MACD(close, window_fast=12, window_slow=26, window_sign=9)
    df['MACD'] = macd.macd()
    df['MACD_Signal'] = macd.macd_signal()
    df['MACD_Diff'] = macd.macd_diff()

    # ADX for trend strength
    adx = ADXIndicator(high, low, close, window=14)
    df['ADX'] = adx.adx()
    df['DI_Plus'] = adx.adx_pos()
    df['DI_Minus'] = adx.adx_neg()

    # Bollinger Bands
    bb = BollingerBands(close, window=20, window_dev=2)
    df['BB_Upper'] = bb.bollinger_hband()
    df['BB_Middle'] = bb.bollinger_mavg()
    df['BB_Lower'] = bb.bollinger_lband()
    
    df['AVG_VOL'] = volume.rolling(20).mean()
    return df

############################
# TREND + MOMENTUM
############################

def compute_trend_factors(df):
    df['EMA50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA200'] = df['Close'].ewm(span=200, adjust=False).mean()
    df['ROC10'] = df['Close'].pct_change(10)
    df['ROC20'] = df['Close'].pct_change(20)
    
    # Trend direction and strength
    df['Price_Above_EMA50'] = (df['Close'] > df['EMA50']).astype(int)
    df['Price_Above_EMA200'] = (df['Close'] > df['EMA200']).astype(int)
    
    return df

############################
# SUPPORT & RESISTANCE
############################

def compute_support_resistance(df):
    """Calculate recent support/resistance levels"""
    recent_high = df['High'].tail(20).max()
    recent_low = df['Low'].tail(20).min()
    
    # Pivot Point
    pivot = (df['High'].iloc[-1] + df['Low'].iloc[-1] + df['Close'].iloc[-1]) / 3
    r1 = (2 * pivot) - df['Low'].iloc[-1]
    s1 = (2 * pivot) - df['High'].iloc[-1]
    
    df['Pivot'] = pivot
    df['R1'] = r1
    df['S1'] = s1
    df['Recent_High'] = recent_high
    df['Recent_Low'] = recent_low
    
    return df

############################
# BREAKOUT DETECTION
############################

def detect_breakout(df):
    """Detect if price is breaking above recent high with volume confirmation"""
    last = df.iloc[-1]
    prev = df.iloc[-2]
    
    recent_high = df['High'].tail(20).max()
    recent_low = df['Low'].tail(20).min()
    
    # Breakout above resistance with volume
    breakout_up = (last['Close'] > recent_high) and (last['Volume'] > last['AVG_VOL'] * 1.5)
    
    # Breakout below support with volume
    breakout_down = (last['Close'] < recent_low) and (last['Volume'] > last['AVG_VOL'] * 1.5)
    
    df['Breakout_Up'] = 1 if breakout_up else 0
    df['Breakout_Down'] = 1 if breakout_down else 0
    
    return df

############################
# DIVERGENCE DETECTION
############################

def detect_divergence(df):
    """Detect price-momentum divergences"""
    last_few = df.tail(5)
    
    # Higher Highs in price but lower highs in RSI = bearish divergence
    price_higher = last_few['Close'].iloc[-1] > last_few['Close'].iloc[-3]
    rsi_lower = last_few['RSI'].iloc[-1] < last_few['RSI'].iloc[-3]
    bearish_div = price_higher and rsi_lower
    
    # Lower lows in price but higher lows in RSI = bullish divergence
    price_lower = last_few['Close'].iloc[-1] < last_few['Close'].iloc[-3]
    rsi_higher = last_few['RSI'].iloc[-1] > last_few['RSI'].iloc[-3]
    bullish_div = price_lower and rsi_higher
    
    df['Bearish_Div'] = 1 if bearish_div else 0
    df['Bullish_Div'] = 1 if bullish_div else 0
    
    return df

############################
# MARKET STRUCTURE (HIGHER HIGHS/LOWS)
############################

def detect_market_structure(df):
    """Detect higher highs/lows for uptrend and lower highs/lows for downtrend"""
    last = df.iloc[-1]
    
    # Get last 5 candles
    recent = df.tail(6)
    
    higher_highs = recent['High'].iloc[-1] > recent['High'].iloc[-2] > recent['High'].iloc[-3]
    higher_lows = recent['Low'].iloc[-1] > recent['Low'].iloc[-2] > recent['Low'].iloc[-3]
    lower_highs = recent['High'].iloc[-1] < recent['High'].iloc[-2] < recent['High'].iloc[-3]
    lower_lows = recent['Low'].iloc[-1] < recent['Low'].iloc[-2] < recent['Low'].iloc[-3]
    
    uptrend = higher_highs and higher_lows
    downtrend = lower_highs and lower_lows
    
    df['Uptrend_Structure'] = 1 if uptrend else 0
    df['Downtrend_Structure'] = 1 if downtrend else 0
    
    return df

############################
# RISK + LIQUIDITY
############################

def compute_risk_factors(df):
    df['Volatility20'] = df['Close'].pct_change().rolling(20).std()
    df['AvgVolume20'] = df['Volume'].rolling(20).mean()
    df['LiquidityScore'] = df['AvgVolume20'] * df['Close']
    
    # Volume trend
    df['Volume_Trend'] = df['Volume'] / df['AvgVolume20']
    
    return df

############################
# ALPHA VS NIFTY
############################

def compute_alpha(stock_df, index_df):
    stock_ret = stock_df['Close'].pct_change(20)
    index_ret = index_df['Close'].pct_change(20)
    stock_df['Alpha'] = stock_ret - index_ret   # <-- FIXED NAME
    return stock_df

############################
# Enhanced Score Stock - With Multiple Factors
############################
def score_stock(df):
    last = df.iloc[-1]
    score = 0

    # ========== PRICE ACTION (3 points max) ==========
    if last['Close'] > last['EMA21']:
        score += 1.5
    if last['Close'] > last['VWAP']:
        score += 1.5

    # ========== EMA CONFIRMATION (3 points max) ==========
    if last['EMA9'] > last['EMA21']:
        score += 1.5
    if last['EMA21'] > last['EMA50']:
        score += 1.5

    # ========== VOLUME CONFIRMATION (3 points max) ==========
    vol_ratio = last['Volume'] / last['AVG_VOL']
    if vol_ratio > 2.0:  # Exceptional volume
        score += 3.0
    elif vol_ratio > 1.5:  # Good volume
        score += 2.0
    elif vol_ratio > 1.2:  # Above average
        score += 1.0

    # ========== RSI ANALYSIS (2 points max) ==========
    if 40 < last['RSI'] < 65:  # Bullish but not overbought
        score += 1.5
    elif last['RSI'] > 65 and last['RSI'] < 85:  # Strong but still tradeable
        score += 1.0

    # ========== ATR & VOLATILITY (1 point max) ==========
    if last['ATR'] > df['ATR'].quantile(0.5):  # Above median ATR
        score += 0.5

    # ========== MACD CONFIRMATION (2 points max) ==========
    if last['MACD'] > last['MACD_Signal']:
        score += 1.5
    if last['MACD_Diff'] > 0:
        score += 0.5

    # ========== STOCHASTIC CONFIRMATION (1 point max) ==========
    if last['Stoch_K'] > last['Stoch_D'] and last['Stoch_K'] < 80:
        score += 1.0

    # ========== ADX TREND STRENGTH (1 point max) ==========
    if last['ADX'] > 25:  # Strong trend
        score += 1.0
    elif last['ADX'] > 20:  # Moderate trend
        score += 0.5

    # ========== DIRECTIONAL INDICATORS (1 point max) ==========
    if last['DI_Plus'] > last['DI_Minus']:
        score += 1.0

    # ========== BREAKOUT BONUS (2 points) ==========
    if last['Breakout_Up'] == 1:
        score += 2.0

    # ========== STRUCTURE CHECK (1 point max) ==========
    if last['Uptrend_Structure'] == 1:
        score += 1.0

    # ========== DIVERGENCE PENALTY (reduce score) ==========
    if last['Bearish_Div'] == 1:
        score -= 2.0

    # ========== BOLLINGER BAND POSITIONING (1 point max) ==========
    bb_position = (last['Close'] - last['BB_Lower']) / (last['BB_Upper'] - last['BB_Lower'])
    if 0.6 < bb_position < 0.9:  # In upper half but not extreme
        score += 0.5

    return score, last

############################
# Compute Factor Score (for broader filters)
############################
def compute_factor_score(last, index_bullish):
    factor_score = 0

    # ================= TREND FACTOR =================
    if last['EMA50'] > last['EMA200']:
        factor_score += 2
    if last['EMA9'] > last['EMA21']:
        factor_score += 2
    if index_bullish:
        factor_score += 1

    # ================= MOMENTUM FACTOR =================
    if last['ROC10'] > 0:
        factor_score += 1.5
    if last['ROC20'] > 0:
        factor_score += 1.5

    # ================= ALPHA FACTOR =================
    if last['Alpha'] > 0:
        factor_score += 2

    # ================= LIQUIDITY FILTER =================
    if last['Volume'] > last['AvgVolume20']:
        factor_score += 1

    # ================= VOLATILITY FILTER (moderate is good) =================
    if 0.01 < last['Volatility20'] < 0.05:
        factor_score += 1

    # ================= ADX STRENGTH =================
    if last['ADX'] > 25:
        factor_score += 2

    # ================= BOLLINGER BAND BREAKOUT =================
    if last['Close'] > last['BB_Upper'] * 0.99:
        factor_score += 1

    return factor_score

############################
# AI Score
############################
def compute_ai_score(ai, news_score):
    ai_score = 0

    # ================= AI BIAS =================
    if ai.get('bias') == 'bullish':
        ai_score += 3
    elif ai.get('bias') == 'bearish':
        ai_score -= 3

    # ================= AI CONFIDENCE =================
    if ai.get('confidence', 0) > 75:
        ai_score += 2
    elif ai.get('confidence', 0) > 50:
        ai_score += 1

    # ================= NEWS SENTIMENT (Higher weight for strong signals) =================
    if news_score > 0.5:
        ai_score += 2
    elif news_score > 0.25:
        ai_score += 1
    elif news_score < -0.5:
        ai_score -= 2
    elif news_score < -0.25:
        ai_score -= 1

    # return ai_score
    return ((ai_score + 4) / 10) * 10


############################
# Detect Indian market Regime
############################
SP500 = "^GSPC"
NASDAQ = "^IXIC"
DXY = "DX-Y.NYB"
CRUDE = "CL=F"
VIX = "^VIX"

def detect_india_regime(index_df):
    index_df['EMA50'] = index_df['Close'].ewm(span=50).mean()
    index_df['EMA200'] = index_df['Close'].ewm(span=200).mean()
    index_df['ROC20'] = index_df['Close'].pct_change(20)

    last = index_df.iloc[-1]

    if last['EMA50'] > last['EMA200'] and last['ROC20'] > 0:
        return "RISK_ON"
    elif last['EMA50'] < last['EMA200'] and last['ROC20'] < 0:
        return "RISK_OFF"
    else:
        return "SIDEWAYS"

############################
# Global market risk
############################

def global_risk_model():
    try:
        sp500 = fetch_data(SP500)
        nasdaq = fetch_data(NASDAQ)
        dxy = fetch_data(DXY)
        crude = fetch_data(CRUDE)
        vix = fetch_data(VIX)

        score = 0

        if sp500['Close'].pct_change(5).iloc[-1] > 0: score += 1
        else: score -= 1

        if nasdaq['Close'].pct_change(5).iloc[-1] > 0: score += 1
        else: score -= 1

        if dxy['Close'].pct_change(5).iloc[-1] > 0: score -= 1
        if crude['Close'].pct_change(5).iloc[-1] > 0: score -= 1
        if vix['Close'].pct_change(5).iloc[-1] > 0: score -= 1

        if score >= 2:
            print('Global Risk On')
            return "GLOBAL_RISK_ON"
        elif score <= -2:
            print('Global Risk Off') 
            return "GLOBAL_RISK_OFF"
        else:
            print('Neutral')
            return "NEUTRAL"

    except:
        return "NEUTRAL"

############################
# Final Market State
############################

def final_market_state(india_regime, global_regime):
    if india_regime == "RISK_ON" and global_regime == "GLOBAL_RISK_ON":
        print('MarketTrend STRONG_BULL')
        return "STRONG_BULL"
    elif india_regime == "RISK_OFF" and global_regime == "GLOBAL_RISK_OFF":
        print('MarketTrend STRONG_BEAR')
        return "STRONG_BEAR"
    else:
        print('MarketTrend MIXED')
        return "MIXED"

############################
# Score Stock
############################

def capital_multiplier(state):
    if state == "STRONG_BULL": return 1.0
    elif state == "MIXED": return 0.5
    else: return 0.25


############################
# RANKING ENGINE
############################
def remove_high_correlation(top_df, threshold=0.8):
    price_data = {}
    for stock in top_df['Stock']:
        try:
            df = fetch_data(stock + ".NS")
            price_data[stock] = df['Close'].pct_change()
        except:
            pass

    if not price_data:
        return top_df
        
    returns = pd.DataFrame(price_data)
    corr = returns.corr()

    selected = []
    for stock in corr.columns:
        if not any(abs(corr[stock][s]) > threshold for s in selected):
            selected.append(stock)

    return top_df[top_df['Stock'].isin(selected)]


############################
# POSITION SIZING (VOLATILITY ADJUSTED)
############################

def position_size(capital, volatility, multiplier):
    target_risk = 0.01  # 1% risk per trade
    adjusted_capital = capital * multiplier
    risk_amount = adjusted_capital * target_risk
    size = risk_amount / (volatility * adjusted_capital) if (volatility * adjusted_capital) > 0 else 0
    return round(size, 4)


# News Read and Sentiment Analysis Function

In [18]:
"""
News Sentiment Engine - Refactored
====================================
Improvements:
- FinBERT model (finance-specific NLP)
- Time-based cache with auto-expiry
- Multiple Indian market news sources (NewsAPI + RSS + Alpha Vantage + FinnHub)
- Weighted scoring across sources
- Batch processing with ThreadPoolExecutor
- Graceful fallbacks
"""

import requests
import hashlib
from datetime import datetime, timedelta
from transformers import pipeline
from concurrent.futures import ThreadPoolExecutor, as_completed
import feedparser  # pip install feedparser
import re

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────


def _get_aliases(stock_name: str) -> list[str]:
    """Return search terms for a stock ticker. Handles .NS/.BSE suffix."""
    clean_ticker = stock_name.replace(".NS", "").replace(".BSE", "")
    return STOCK_ALIASES.get(clean_ticker.upper(), [clean_ticker])


# ─────────────────────────────────────────────
# HTML CLEANER
# ─────────────────────────────────────────────

def _clean_text(text: str) -> str:
    """Strip HTML tags and entities from RSS/news text."""
    text = re.sub(r'<[^>]+>', ' ', text)           # <img ...>, <a ...>, etc.
    text = re.sub(r'&#\d+;', "'", text)             # &#39; → '
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&[a-z]+;', ' ', text)           # &nbsp; etc.
    text = re.sub(r'\s+', ' ', text)                # collapse whitespace
    return text.strip()


# ─────────────────────────────────────────────
# LOAD FINBERT (once at startup)
# ─────────────────────────────────────────────

print("[SENTIMENT] Loading FinBERT model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",   # Finance-trained model
    device=-1,
    truncation=True,
    max_length=512
)
print("[SENTIMENT] FinBERT loaded.")


# ─────────────────────────────────────────────
# TIME-BASED CACHE (replaces @lru_cache)
# ─────────────────────────────────────────────

_sentiment_cache: dict = {}

def _cache_key(stock_name: str) -> str:
    return hashlib.md5(stock_name.lower().encode()).hexdigest()

def _get_cache(stock_name: str):
    key = _cache_key(stock_name)
    if key in _sentiment_cache:
        result, timestamp = _sentiment_cache[key]
        if datetime.now() - timestamp < timedelta(minutes=CACHE_EXPIRY_MINUTES):
            print(f"[CACHE HIT] {stock_name}")
            return result
    return None

def _set_cache(stock_name: str, result):
    key = _cache_key(stock_name)
    _sentiment_cache[key] = (result, datetime.now())


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def _score_text(text: str) -> float:
    """Run FinBERT on a single text. Returns score in [-1, +1]."""
    text = text[:SENTIMENT_TRUNCATE].strip()
    if not text:
        return 0.0
    try:
        out = sentiment_model([text])[0]
        label = out["label"].lower()   # 'positive', 'negative', 'neutral'
        score = out["score"]
        if label == "positive":
            return round(score, 3)
        elif label == "negative":
            return round(-score, 3)
        else:
            return 0.0
    except Exception as e:
        print(f"[FINBERT ERROR] {e}")
        return 0.0

def _combine_title_desc(article: dict) -> str:
    title = article.get("title") or ""
    desc  = article.get("description") or ""
    return f"{title}. {desc}".strip()


# ─────────────────────────────────────────────
# SOURCE 1: NewsAPI (General + India filter)
# ─────────────────────────────────────────────

def _fetch_newsapi(stock_name: str) -> list[str]:
    try:
        aliases   = _get_aliases(stock_name)
        # Build OR query: "Reliance Industries" OR "Reliance"
        query     = " OR ".join(f'"{a}"' for a in aliases)
        url = (
            f"https://newsapi.org/v2/everything"
            f"?q={requests.utils.quote(query)}"
            f"&pageSize={MAX_ARTICLES}"
            f"&language=en"
            f"&sortBy=publishedAt"
            f"&apiKey={NEWS_API_KEY}"
        )
        r = requests.get(url, timeout=5)
        if r.status_code != 200:
            print(f"[NEWSAPI] HTTP {r.status_code}")
            return []
        data     = r.json()
        articles = data.get("articles", [])

        # Relevance filter: at least one alias must appear in title/description
        alias_lower = [a.lower() for a in aliases]
        filtered = []
        for a in articles:
            text = _combine_title_desc(a).lower()
            if any(alias in text for alias in alias_lower):
                filtered.append(_clean_text(_combine_title_desc(a)))

        return filtered
    except Exception as e:
        print(f"[NEWSAPI ERROR] {e}")
        return []


# ─────────────────────────────────────────────
# SOURCE 2: RSS Feeds (Indian Market News)
# ─────────────────────────────────────────────

INDIAN_RSS_FEEDS = [
    "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",   # ET Markets
    "https://www.moneycontrol.com/rss/marketreports.xml",                     # Moneycontrol
    "https://feeds.feedburner.com/ndtvprofit-latest",                          # NDTV Profit
]

def _fetch_rss(stock_name: str) -> list[str]:
    """Search RSS feeds for relevant headlines using aliases."""
    results     = []
    alias_lower = [a.lower() for a in _get_aliases(stock_name)]
    try:
        for feed_url in INDIAN_RSS_FEEDS:
            feed = feedparser.parse(feed_url)
            for entry in feed.entries[:30]:  # scan more entries
                title   = _clean_text(entry.get("title", ""))
                summary = _clean_text(entry.get("summary", ""))
                combined = f"{title}. {summary}"
                # Only include if at least one alias appears in first 100 chars (main topic)
                if any(alias in combined[:100].lower() for alias in alias_lower):
                    results.append(combined[:SENTIMENT_TRUNCATE])
    except Exception as e:
        print(f"[RSS ERROR] {e}")
    return results[:MAX_ARTICLES]


# ─────────────────────────────────────────────
# SOURCE 3: Alpha Vantage News Sentiment API
# ─────────────────────────────────────────────

def _fetch_alphavantage(stock_name: str) -> list[str]:
    """
    Try multiple ticker formats for Alpha Vantage.
    NSE stocks: BSE:RELIANCE or just RELIANCE sometimes works.
    """
    ticker_formats = [
        f"BSE:{stock_name.upper()}",
        stock_name.upper(),
    ]
    for ticker in ticker_formats:
        try:
            url = (
                f"https://www.alphavantage.co/query"
                f"?function=NEWS_SENTIMENT"
                f"&tickers={ticker}"
                f"&limit={MAX_ARTICLES}"
                f"&apikey={ALPHA_VANTAGE_KEY}"
            )
            r    = requests.get(url, timeout=6)
            data = r.json()
            feed = data.get("feed", [])
            if feed:
                texts = []
                for item in feed:
                    title   = item.get("title", "")
                    summary = item.get("summary", "")
                    texts.append(_clean_text(f"{title}. {summary}")[:SENTIMENT_TRUNCATE])
                print(f"[ALPHAVANTAGE] {ticker} → {len(texts)} articles")
                return texts
        except Exception as e:
            print(f"[ALPHAVANTAGE ERROR] {ticker}: {e}")
    return []


# ─────────────────────────────────────────────
# SOURCE 4: FinnHub Company News API
# ─────────────────────────────────────────────

def _fetch_finnhub(stock_name: str) -> list[str]:
    """
    Fetch company news from FinnHub.
    NSE stocks format: RELIANCE.NS or just RELIANCE
    Free tier: 60 calls/minute
    """
    ticker_formats = [
        f"{stock_name.upper()}.NS",   # NSE format
        f"{stock_name.upper()}.BSE",  # BSE format
        stock_name.upper(),            # Try without suffix
    ]
    
    from datetime import datetime, timedelta
    date_to   = datetime.now().strftime('%Y-%m-%d')
    date_from = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
    
    for ticker in ticker_formats:
        try:
            url = (
                f"https://finnhub.io/api/v1/company-news"
                f"?symbol={ticker}"
                f"&from={date_from}"
                f"&to={date_to}"
                f"&token={FINNHUB_API_KEY}"
            )
            r = requests.get(url, timeout=5)
            if r.status_code != 200:
                continue
                
            data = r.json()
            if data and len(data) > 0:
                texts = []
                alias_lower = [a.lower() for a in _get_aliases(stock_name)]
                
                for item in data[:20]:  # scan more to filter
                    headline = _clean_text(item.get("headline", ""))
                    summary  = _clean_text(item.get("summary", ""))
                    combined = f"{headline}. {summary}"
                    
                    # Relevance check: stock name in first 100 chars
                    if any(alias in combined[:100].lower() for alias in alias_lower):
                        texts.append(combined[:SENTIMENT_TRUNCATE])
                        
                if texts:
                    print(f"[FINNHUB] {ticker} → {len(texts[:MAX_ARTICLES])} articles")
                    return texts[:MAX_ARTICLES]
                    
        except Exception as e:
            print(f"[FINNHUB ERROR] {ticker}: {e}")
            
    return []


# ─────────────────────────────────────────────
# MAIN: MULTI-SOURCE SENTIMENT FUNCTION
# ─────────────────────────────────────────────

def fetch_news_sentiment(stock_name: str) -> tuple[float, str, dict]:
    """
    Fetch and analyze news sentiment from multiple sources.

    Returns:
        (final_score, status, details)
        final_score : float in [-1.0, +1.0]
                      positive = bullish, negative = bearish
        status      : str  (e.g. "news_ok", "no_news", "exception")
        details     : dict with per-source breakdown
    """

    # ── Cache check ──
    cached = _get_cache(stock_name)
    if cached:
        return cached

    # ── Fetch from all sources in parallel ──
    sources = {
        "newsapi":       lambda: _fetch_newsapi(stock_name),
        "rss":           lambda: _fetch_rss(stock_name),
        # "alphavantage":  lambda: _fetch_alphavantage(stock_name),
        "finnhub":       lambda: _fetch_finnhub(stock_name),
    }

    source_texts: dict[str, list[str]] = {}
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(fn): name for name, fn in sources.items()}
        for future in as_completed(futures, timeout=12):
            name = futures[future]
            try:
                source_texts[name] = future.result()
            except Exception as e:
                print(f"[SOURCE ERROR: {name}] {e}")
                source_texts[name] = []

    # ── Score each source ──
    source_weights = {
        "newsapi": 0.25, 
        "rss": 0.35, 
        "alphavantage": 0.15,
        "finnhub": 0.25
    }
    source_scores  = {}
    details        = {}

    for source, texts in source_texts.items():
        if not texts:
            details[source] = {"score": 0.0, "articles": 0, "status": "no_news"}
            continue

        scores = [_score_text(t) for t in texts]
        avg    = round(sum(scores) / len(scores), 3)

        source_scores[source] = avg
        details[source] = {
            "score":    avg,
            "articles": len(texts),
            "status":   "ok",
            "headlines": [t[:100] for t in texts[:3]]  # preview only
        }

    # ── Weighted final score ──
    if not source_scores:
        result = (0.0, "no_news", details)
        _set_cache(stock_name, result)
        return result

    total_weight = sum(source_weights[s] for s in source_scores)
    final_score  = sum(
        source_scores[s] * source_weights[s]
        for s in source_scores
    ) / total_weight

    final_score = round(final_score, 3)
    status      = "news_ok"

    result = (final_score, status, details)
    _set_cache(stock_name, result)

    print(f"[SENTIMENT] {stock_name}: score={final_score}, sources={list(source_scores.keys())}")
    return result


# ─────────────────────────────────────────────
# BATCH PROCESSING (for scanning multiple stocks)
# ─────────────────────────────────────────────

def batch_sentiment(stock_list: list[str], max_workers: int = 4) -> dict:
    """
    Run sentiment analysis on multiple stocks in parallel.

    Returns:
        dict of { stock_name: (score, status, details) }
    """
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_news_sentiment, stock): stock for stock in stock_list}
        for future in as_completed(futures):
            stock = futures[future]
            try:
                results[stock] = future.result()
            except Exception as e:
                print(f"[BATCH ERROR: {stock}] {e}")
                results[stock] = (0.0, "exception", {})
    return results


# ─────────────────────────────────────────────
# USAGE EXAMPLE
# ─────────────────────────────────────────────

# if __name__ == "__main__":

#     all_results = batch_sentiment(STOCKS)
#     for stock, (score, status, _) in all_results.items():
#         print(f"{stock:12s} → score={score:+.3f}  [{status}]")

[SENTIMENT] Loading FinBERT model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SENTIMENT] FinBERT loaded.


# AI Ollama local model to add additional filter on stock

In [19]:
import subprocess
import json


def ai_filter(stock_name, data_row, news_score):
    """
    Enhanced AI filter with:
    - News sentiment priority
    - Technical confluence scoring
    - Strong JSON validation
    - Risk override logic
    """

    # Normalize news score to -1 to +1 range if needed
    try:
        news_score = float(news_score)
    except:
        news_score = 0.0

    prompt = f"""
You are a professional Indian intraday trading assistant AI.

IMPORTANT RULES:
1. News sentiment has HIGH priority.
2. If NewsSentimentScore > 0.6 → Strong bullish bias unless extreme bearish technicals.
3. If NewsSentimentScore < -0.6 → Strong bearish bias unless extreme bullish technicals.
4. If NewsSentimentScore between -0.3 and 0.3 → rely mostly on technical indicators.
5. Increase confidence when:
   - Price > VWAP (bullish intraday strength)
   - EMA9 > EMA21 (short term bullish crossover)
   - RSI between 55-70 (healthy bullish)
   - Volume above average (assume strong if Volume high)
6. Decrease confidence when signals conflict.
7. Confidence must reflect strength of confluence.

Return ONLY valid JSON.
No explanation.
No markdown.

Stock: {stock_name}
Price: {data_row['Close']}
VWAP: {data_row['VWAP']}
RSI: {data_row['RSI']}
Volume: {data_row['Volume']}
EMA9: {data_row['EMA9']}
EMA21: {data_row['EMA21']}
ATR: {data_row['ATR']}
NewsSentimentScore: {news_score}

JSON FORMAT:
{{
  "bias": "bullish/bearish/neutral",
  "confidence": 0-100,
  "reason": "short technical + news reason"
}}
"""

    try:
        result = subprocess.run(
            ["ollama", "run", OLLAMA_MODEL],
            input=prompt.encode("utf-8"),
            capture_output=True,
            timeout=30
        )

        if result.returncode != 0:
            print("[OLLAMA ERROR]", result.stderr.decode())
            return {"bias": "neutral", "confidence": 0, "reason": "ollama_error"}

        output = result.stdout.decode("utf-8").strip()

        if not output:
            print("[OLLAMA EMPTY OUTPUT]")
            return {"bias": "neutral", "confidence": 0, "reason": "empty_output"}

        # Extract JSON safely
        start = output.find("{")
        end = output.rfind("}")
        if start == -1 or end == -1:
            print("[BAD OUTPUT]", output[:300])
            return {"bias": "neutral", "confidence": 0, "reason": "no_json"}

        json_text = output[start:end+1]
        data = json.loads(json_text)

        # -----------------------
        # Strict Validation Layer
        # -----------------------

        required_keys = {"bias", "confidence", "reason"}
        if not required_keys.issubset(data.keys()):
            return {"bias": "neutral", "confidence": 0, "reason": "bad_json_structure"}

        # Normalize bias
        if data["bias"] not in ["bullish", "bearish", "neutral"]:
            data["bias"] = "neutral"

        # Clamp confidence
        try:
            data["confidence"] = int(data["confidence"])
        except:
            data["confidence"] = 0

        data["confidence"] = max(0, min(100, data["confidence"]))

        # ---------------------------------
        # 🔥 News Override Risk Layer
        # ---------------------------------

        # Strong positive news override
        if news_score > 0.75 and data["bias"] == "bearish":
            data["bias"] = "bullish"
            data["confidence"] = max(data["confidence"], 70)
            data["reason"] += " | strong positive news override"

        # Strong negative news override
        if news_score < -0.75 and data["bias"] == "bullish":
            data["bias"] = "bearish"
            data["confidence"] = max(data["confidence"], 70)
            data["reason"] += " | strong negative news override"

        print(f"[AI RESPONSE] {stock_name}: {data}")
        return data

    except subprocess.TimeoutExpired:
        print("[OLLAMA TIMEOUT]")
        return {"bias": "neutral", "confidence": 0, "reason": "timeout"}

    except json.JSONDecodeError as e:
        print("[JSON ERROR]", str(e))
        print("[RAW OUTPUT]", output[:300])
        return {"bias": "neutral", "confidence": 0, "reason": "json_error"}

    except Exception as e:
        print("[AI FILTER ERROR]", str(e))
        return {"bias": "neutral", "confidence": 0, "reason": "exception"}


# Calling All the Funcation to Generate the output

In [20]:
from tabulate import tabulate
import shutil
import pandas as pd
import numpy as np
import time

INDEX_SYMBOL = "^NSEI"  # NIFTY Index

# ==============================================
# PIPELINE
# ==============================================

results = []

print("[INFO] Fetching & analyzing index...")
index_df = fetch_data(INDEX_SYMBOL)
index_df = compute_indicators(index_df)
index_df = compute_trend_factors(index_df)

india_regime = detect_india_regime(index_df)
global_regime = global_risk_model()
market_state = final_market_state(india_regime, global_regime)
cap_multiplier = capital_multiplier(market_state)

index_bullish = index_df.iloc[-1]['EMA50'] > index_df.iloc[-1]['EMA200']

print(f"[INFO] Market State: {market_state} | India: {india_regime} | Global: {global_regime}")
print(f"[INFO] Processing {len(STOCKS)} stocks...\n")

for idx, stock in enumerate(STOCKS):
    try:
        print(f"[{idx+1}/{len(STOCKS)}] Analyzing {stock}...", end=" ")
        
        df = fetch_data(stock)
        df = compute_indicators(df)
        df = compute_trend_factors(df)
        df = compute_risk_factors(df)
        df = compute_support_resistance(df)
        df = detect_breakout(df)
        df = detect_divergence(df)
        df = detect_market_structure(df)
        df = compute_alpha(df, index_df)

        # ================= SCORING =================
        base_score, last = score_stock(df)
        factor_score = compute_factor_score(last, index_bullish)

        # ================= AI + NEWS =================
        news_score, news_status, news_details = fetch_news_sentiment(stock.replace('.NS',''))
        ai = ai_filter(stock.replace('.NS',''), last, news_score)
        ai_score = compute_ai_score(ai, news_score)
        time.sleep(0.5)

        # ================= FINAL SCORING WITH BETTER WEIGHTS =================
        # Base Score: 40% (technical fundamentals)
        # Factor Score: 35% (trend & confluence)
        # AI Score: 25% (sentiment + AI analysis)
        # This weighting gives more weight to technical analysis while keeping sentiment
        final_score = (base_score * 0.40 + factor_score * 0.35 + ai_score * 0.25)

        # CONFIDENCE CHECK: Only include stocks with minimum confluence
        confidence_level = (last.get('ADX', 0) + ai.get('confidence', 0) + (abs(news_score) * 100)) / 3
        
        # Risk filters
        is_extreme_rsi = last['RSI'] > 85 or last['RSI'] < 15
        has_divergence = last.get('Bearish_Div', 0) == 1
        has_volume = last['Volume'] > last['AVG_VOL'] * 1.2
        
        results.append({
            "Stock": stock.replace('.NS',''),
            "FinalScore": round(final_score, 2),
            "BaseScore": round(base_score, 2),
            "FactorScore": round(factor_score, 2),
            "News_Score": round(news_score, 2),
            "AI_Score": round(ai_score, 2),
            "AI_Bias": ai['bias'],
            "AI_Confidence": ai['confidence'],
            "Alpha": round(last['Alpha'], 4),
            "ADX": round(last['ADX'], 2),
            "RSI": round(last['RSI'], 2),
            "MACD_Signal": round(last['MACD_Diff'], 4),
            "Volatility": round(last['Volatility20'], 4),
            "Liquidity": int(last['LiquidityScore']),
            "Price": round(last['Close'], 2),
            "Volume_Ratio": round(last['Volume'] / last['AVG_VOL'], 2),
            "Trend_Structure": "UP" if last.get('Uptrend_Structure', 0) == 1 else ("DOWN" if last.get('Downtrend_Structure', 0) == 1 else "MIXED"),
            "Breakout": "YES" if last.get('Breakout_Up', 0) == 1 else "NO",
            "Confidence": round(confidence_level, 2),
            "High_Volume": "YES" if has_volume else "NO",
            "Extreme_RSI": "YES" if is_extreme_rsi else "NO",
            "Bearish_Div": "YES" if has_divergence else "NO",
        })
        
        print("✓")

    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        continue


[INFO] Fetching & analyzing index...
Neutral
MarketTrend MIXED
[INFO] Market State: MIXED | India: SIDEWAYS | Global: NEUTRAL
[INFO] Processing 50 stocks...

[1/50] Analyzing ADANIENT.NS... [SENTIMENT] ADANIENT: score=0.499, sources=['newsapi']
[AI RESPONSE] ADANIENT: {'bias': 'bullish', 'confidence': 75, 'reason': 'NewsSentimentScore is positive and above 0.3. Price above VWAP, RSI is high, EMA9 crossover, and volume is above average. Confluence is strong.'}
✓
[2/50] Analyzing ADANIPORTS.NS... [SENTIMENT] ADANIPORTS: score=0.766, sources=['newsapi']
[AI RESPONSE] ADANIPORTS: {'bias': 'bullish', 'confidence': 92, 'reason': 'Strong bullish bias due to high NewsSentimentScore (0.766) and positive technical indicators: Price above VWAP, EMA9 > EMA21, RSI above 60, and high volume. Confluence of factors strengthens the bullish outlook.'}
✓
[3/50] Analyzing APOLLOHOSP.NS... [SENTIMENT] APOLLOHOSP: score=0.0, sources=['newsapi']
[AI RESPONSE] APOLLOHOSP: {'bias': 'bullish', 'confidence': 75,

$TATAMOTORS.NS: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


✗ Error: index 13 is out of bounds for axis 0 with size 0
[44/50] Analyzing TATASTEEL.NS... [SENTIMENT] TATASTEEL: score=-0.163, sources=['newsapi']
[AI RESPONSE] TATASTEEL: {'bias': 'neutral', 'confidence': 65, 'reason': 'RSI is above 60 indicating bullish momentum, Price is above VWAP, EMA9 is above EMA21, but NewsSentimentScore is slightly negative. Relying on technical indicators for now.'}
✓
[45/50] Analyzing TCS.NS... [SENTIMENT] TCS: score=0.0, sources=['newsapi', 'rss']
[AI RESPONSE] TCS: {'bias': 'bullish', 'confidence': 75, 'reason': 'NewsSentimentScore is neutral, but VWAP is slightly below current price, EMA9 is above EMA21, RSI is near neutral, and volume is high.  Overall, a bullish bias with moderate confidence due to the technical setup and news neutrality.'}
✓
[46/50] Analyzing TECHM.NS... [AI RESPONSE] TECHM: {'bias': 'bullish', 'confidence': 85, 'reason': 'NewsSentimentScore is 0.0, Price > VWAP, RSI is high, EMA9 > EMA21, and Volume is above average. Strong bullish 

# Display Top Stocks for Intraday Trading

In [21]:
terminal_width = shutil.get_terminal_size().columns
ranked = pd.DataFrame(results).sort_values(by="FinalScore", ascending=False)

print(f"[INFO] Candidate universe: {len(ranked)} stocks")

# APPLY FILTERS FOR QUALITY STOCKS (relaxed intraday version)
print("\n" + "=" * terminal_width)
print("📊 APPLYING QUALITY FILTERS...".center(terminal_width))
print("=" * terminal_width)

score_min = 12
adx_min = 12
volume_multiplier = 1.1

# Filter 1: Minimum score threshold
ranked = ranked[ranked['FinalScore'] >= score_min]
print(f"✓ Score Filter: {len(ranked)} stocks remaining (FinalScore >= {score_min})")

# Filter 2: No extreme RSI (avoid overbought/oversold traps)
ranked = ranked[ranked['Extreme_RSI'] == 'NO']
print(f"✓ RSI Filter: {len(ranked)} stocks remaining (Extreme_RSI == NO)")

# Filter 3: No major bearish divergences
ranked = ranked[ranked['Bearish_Div'] == 'NO']
print(f"✓ Divergence Filter: {len(ranked)} stocks remaining (Bearish_Div == NO)")

# Filter 4: Minimum ADX (ensure trend component)
ranked = ranked[ranked['ADX'] >= adx_min]
print(f"✓ ADX Filter: {len(ranked)} stocks remaining (ADX >= {adx_min})")

# Filter 5: Positive AI bias during strong bull markets
if market_state == "STRONG_BULL":
    ranked = ranked[ranked['AI_Bias'] != 'bearish']
    print(f"✓ Sentiment Filter (STRONG_BULL): {len(ranked)} stocks remaining")

# Filter 6: Volume confirmation (relative to avg)
ranked = ranked[ranked['Volume_Ratio'] >= volume_multiplier]
print(f"✓ Volume Filter: {len(ranked)} stocks remaining (Volume_Ratio >= {volume_multiplier})")

# Relaxing logic when final set is empty
if ranked.empty:
    print("⚠️ No stocks passed all filters. Applying relaxed backup filter set...")
    ranked = pd.DataFrame(results).sort_values(by="FinalScore", ascending=False)
    ranked = ranked[ranked['FinalScore'] >= 10]
    ranked = ranked[ranked['Extreme_RSI'] == 'NO']
    ranked = ranked[ranked['Bearish_Div'] == 'NO']
    ranked = ranked[ranked['ADX'] >= 10]
    ranked = ranked[ranked['Volume_Ratio'] >= 1.0]
    if market_state == "STRONG_BULL":
        ranked = ranked[ranked['AI_Bias'] != 'bearish']
    print(f"⚠️ Relaxed filters result: {len(ranked)} stocks")

# Final candidate set
if ranked.empty:
    print("⚠️ Still no candidates. Showing top 10 raw scored symbols for manual review.")
    ranked = pd.DataFrame(results).sort_values(by="FinalScore", ascending=False).head(10)

# Get top stocks
top10 = ranked.head(15)
if len(top10) > 0:
    top10 = remove_high_correlation(top10)

print("\n" + "=" * terminal_width)
print("🏦 INTRADAY TRADING CANDIDATES - ENHANCED ANALYSIS".center(terminal_width))
print("=" * terminal_width)

# Display key columns
display_cols = ['Stock', 'FinalScore', 'BaseScore', 'FactorScore', 'News_Score', 
                'AI_Score', 'AI_Bias', 'RSI', 'ADX', 'Price', 'Volume_Ratio', 
                'Trend_Structure', 'Breakout', 'High_Volume', 'Confidence']

print(tabulate(
    top10[display_cols],
    headers="keys",
    tablefmt="fancy_grid",
    showindex=False,
    floatfmt=".2f"
))

print("\n" + "=" * terminal_width)
print("📈 DETAILED ANALYSIS FOR TOP 3 STOCKS".center(terminal_width))
print("=" * terminal_width)

for i, (_, row) in enumerate(top10.head(3).iterrows(), 1):
    print(f"\n#{i} - {row['Stock']} | Price: ₹{row['Price']}")
    print("-" * 60)
    print(f"  Final Score: {row['FinalScore']} | Base: {row['BaseScore']} | " +
          f"Factor: {row['FactorScore']} | AI: {row['AI_Score']}")
    print(f"  Technical: RSI={row['RSI']:.1f} | ADX={row['ADX']:.1f} | " +
          f"MACD={'Bullish' if row['MACD_Signal'] > 0 else 'Bearish'}")
    print(f"  Volume: {row['Volume_Ratio']}x avg | Volatility: {row['Volatility']:.4f}")
    print(f"  Sentiment: {row['News_Score']:+.2f} | AI: {row['AI_Bias']} " +
          f"(Confidence: {row['AI_Confidence']}%)")
    print(f"  Chart Pattern: {row['Trend_Structure']} Trend | Breakout: {row['Breakout']}")
    print(f"  Overall Confidence: {row['Confidence']:.1f}%")

print("\n" + "=" * terminal_width)
print("💰 VOLATILITY-ADJUSTED POSITION SIZING".center(terminal_width))
print("=" * terminal_width)

capital = 1000000  # Example 10L capital
size_table = []

for _, row in top10.head(10).iterrows():
    size = position_size(capital, row['Volatility'], cap_multiplier)
    risk_amount = (capital * cap_multiplier) * 0.01  # 1% risk
    
    size_table.append({
        'Stock': row['Stock'],
        'Price': row['Price'],
        'Position_Qty': size,
        'Confidence': f"{row['Confidence']:.1f}%",
        'Risk_Per_Trade': f"₹{risk_amount:,.0f}",
        'Market_State': market_state
    })

print(tabulate(
    size_table,
    headers="keys",
    tablefmt="fancy_grid",
    showindex=False
))

print("\n" + "=" * terminal_width)
print("KEY IMPROVEMENTS IN THIS VERSION:".center(terminal_width))
print("=" * terminal_width)
print("""
✅ ADDED TECHNICAL INDICATORS:
   • ADX (Trend Strength) - Only trades with ADX >= 15
   • MACD (Trend Confirmation) - Checks MACD > Signal
   • Stochastic Oscillator - Momentum confirmation
   • Bollinger Bands - Volatility breakout zones
   • Support/Resistance - Recent pivot points

✅ ADDED PATTERN RECOGNITION:
   • Breakout Detection - Price above 20-day high + volume
   • Divergence Detection - Price vs RSI divergences
   • Market Structure - Higher Highs/Lows detection
   • Volume Profile - 1.2x to 2.0x avg volume checks

✅ IMPROVED FILTERS:
   • RSI Extremes Filter - Avoid overbought/oversold traps
   • ADX Trend Filter - Only trending, not sideways markets
   • Volume Confirmation - Must have above-average volume
   • Divergence Filter - Avoid bearish divergences
   • Correlation Filter - Reduce portfolio correlation

✅ BETTER SCORING WEIGHTS:
   • Base Score: 40% (technical fundamentals)
   • Factor Score: 35% (trend & confluence)
   • AI Score: 25% (sentiment analysis)
   • Overall Confidence Score (0-100%)

✅ MARKET STATE AWARE:
   • Different filters for STRONG_BULL vs MIXED vs STRONG_BEAR markets
   • Position sizing adjusted to market regime
   • Sentiment filtering active in bullish markets

⚠️ RECOMMENDATION:
   • Paper trade the top 3-5 stocks for 2-3 days
   • Monitor actual market movement vs predictions
   • Adjust weights if consistently wrong direction
   • Check if Entry/SL/Target levels match candle patterns
""")

print("\n✅ ENHANCED ANALYSIS COMPLETE")
print("⚠️ Use for research & educational purposes only.")

[INFO] Candidate universe: 49 stocks

                         📊 APPLYING QUALITY FILTERS...                          
✓ Score Filter: 6 stocks remaining (FinalScore >= 12)
✓ RSI Filter: 6 stocks remaining (Extreme_RSI == NO)
✓ Divergence Filter: 6 stocks remaining (Bearish_Div == NO)
✓ ADX Filter: 6 stocks remaining (ADX >= 12)
✓ Volume Filter: 6 stocks remaining (Volume_Ratio >= 1.1)

               🏦 INTRADAY TRADING CANDIDATES - ENHANCED ANALYSIS                
╒════════════╤══════════════╤═════════════╤═══════════════╤══════════════╤════════════╤═══════════╤═══════╤═══════╤═════════╤════════════════╤═══════════════════╤════════════╤═══════════════╤══════════════╕
│ Stock      │   FinalScore │   BaseScore │   FactorScore │   News_Score │   AI_Score │ AI_Bias   │   RSI │   ADX │   Price │   Volume_Ratio │ Trend_Structure   │ Breakout   │ High_Volume   │   Confidence │
╞════════════╪══════════════╪═════════════╪═══════════════╪══════════════╪════════════╪═══════════╪═══════╪═══════╪